# Chapter 5-4. 백테스팅 심화 & 워크포워드 분석 (Walk-Forward Analysis)

> **KDT AI 퀀트 · 백테스트/성과검증 파트 심화 실습**

앞선 실습(기본 백테스팅 툴 개발)에서 우리는 삼성전자 데이터로
**"5일 최저가 + 20일 이평 하회 시 매수 → 3일 후 매도"** 라는 평균회귀(Mean-Reversion) 전략을
`for` 루프로 백테스팅했습니다.

이번 챕터의 목표는 크게 세 가지입니다.

| 단계 | 내용 |
|------|------|
| **1. 리팩토링** | 매번 복붙하던 백테스팅/성과지표 코드를 **재사용 가능한 함수**로 정리 |
| **2. 응용 실습** | 전략 변형·손절·거래비용·멀티종목 등 **직접 손대는 실습과제** |
| **3. 검증 방법론** | **In-Sample / Out-of-Sample 분리**, 과최적화(Overfitting) 시연, **워크포워드 분석** |

> ### ⚠️ 왜 이 챕터가 제일 중요한가?
> 백테스팅에서 예쁜 우상향 곡선을 만드는 것은 **누구나 할 수 있습니다.**
> 파라미터를 몇 개만 바꿔가며 과거 데이터에 맞추면(curve-fitting) 수익률은 얼마든지 올라갑니다.
> 하지만 **그 전략이 미래에도 통할지**는 완전히 다른 문제입니다.
> 이 간극을 메우는 표준 방법론이 바로 **워크포워드 테스트**입니다.


---
## 0. 환경 설정

Colab에서 바로 실행되도록 데이터는 `FinanceDataReader`로 내려받습니다.
(원본 실습의 `005930.parquet` 파일이 없어도 동작합니다.)


In [ ]:
# Colab 환경 세팅
!pip install -q finance-datareader

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
# 그래프 축 라벨은 한글 폰트 깨짐을 피하기 위해 영문으로 표기합니다.


---
## 1. 데이터 준비

원본과 동일한 삼성전자(005930)를 사용하되, **워크포워드 분석을 위해 기간을 넉넉히**(2015년~) 받습니다.
기간이 길수록 In-Sample / Out-of-Sample 구간을 여러 개 나눌 수 있습니다.


In [ ]:
# 삼성전자 일봉 데이터 (컬럼명을 원본 실습과 동일하게 소문자 'close'로 통일)
raw = fdr.DataReader('005930', '2015')
df = raw.rename(columns=str.lower)[['open', 'high', 'low', 'close', 'volume']].copy()

print(f"기간: {df.index[0].date()} ~ {df.index[-1].date()}  (총 {len(df)} 거래일)")
df.tail()


---
## 2. 백테스팅 로직 함수화 (Refactoring)

원본 실습에서는 아래 코드를 **셀마다 통째로 복사**했습니다.

```python
holding_cash = 1_000_000
position = 0
...
for idx, data in d.iterrows():
    if (data['close'] < data['20d_mean']) and (data['close'] == data['5d_min']):
        ...
```

파라미터(이평 기간, 최저가 기간, 보유일수, 슬리피지)를 바꿔 실험하려면
**함수로 감싸는 것**이 필수입니다. 아래 `run_backtest()` 는 원본 로직을 그대로 옮기되,
전략 파라미터를 인자로 받고 **일별 포트폴리오 가치(equity curve)** 를 반환합니다.

> 📌 **Look-ahead(미래참조) 주의**: 지표(이동평균·롤링 최저가)는 전체 구간에서 미리 계산합니다.
> `rolling()` 은 항상 **과거 방향**만 보기 때문에, 계산 후 특정 구간만 잘라 시뮬레이션해도
> 미래 정보가 새지 않습니다. 이 성질이 뒤의 워크포워드 구현의 핵심입니다.


In [ ]:
def run_backtest(df, ma_window=20, min_window=5, holding_days=3,
                 slippage=0.004, init_cash=1_000_000, start=None, end=None):
    """평균회귀 전략 백테스트.

    매수: 종가가 (min_window)일 최저가 이고 && (ma_window)일 이평 아래일 때 1주 매수
    매도: 마지막 매수 후 holding_days일 경과 시 전량 매도(슬리피지 반영)

    Returns
    -------
    equity : pd.Series   # start~end 구간의 일별 총 포트폴리오 가치
    """
    data = df.copy()
    # 지표는 전체 구간에서 계산 (rolling은 과거만 참조 → look-ahead 없음)
    data['ma'] = data['close'].rolling(ma_window).mean()
    data['roll_min'] = data['close'].rolling(min_window).min()

    sim = data.loc[start:end]  # 시뮬레이션 구간만 슬라이스

    cash, position, holding = init_cash, 0, 0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        # 지표가 아직 없으면(워밍업) 매매 없이 평가액만 기록
        if not (np.isnan(row['ma']) or np.isnan(row['roll_min'])):
            # 매수 조건
            if (price < row['ma']) and (price == row['roll_min']):
                if cash > price:
                    position += 1
                    cash -= price
                    holding = 0
            # 매도 조건 (마지막 매수 holding_days일 후)
            if position > 0 and holding == holding_days:
                cash += position * price * (1 - slippage)
                position = 0
        if position > 0:
            holding += 1
        equity.append(cash + position * price)

    return pd.Series(equity, index=sim.index, name='equity')


### 성과지표 함수화

원본에서 반복 계산하던 **총수익률 / CAGR / Sharpe / MDD** 도 한 함수로 묶습니다.


In [ ]:
def performance(equity, periods_per_year=250, rf=0.0):
    """equity curve로부터 핵심 성과지표를 계산해 dict로 반환."""
    equity = pd.Series(equity).reset_index(drop=True).astype(float)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    years = len(equity) / periods_per_year
    cagr = (equity.iloc[-1] / equity.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan

    daily = equity.pct_change().dropna()
    # 연율화 Sharpe (초과수익 기준)
    excess = daily - rf / periods_per_year
    sharpe = (excess.mean() / daily.std()) * np.sqrt(periods_per_year) if daily.std() > 0 else np.nan

    dd = equity / equity.cummax() - 1
    mdd = dd.min()

    return {'total_return': total_return, 'cagr': cagr, 'sharpe': sharpe, 'mdd': mdd}


def summary(name, equity):
    m = performance(equity)
    print(f"[{name}]")
    print(f"  총수익률 : {m['total_return']*100:7.2f}%")
    print(f"  CAGR     : {m['cagr']*100:7.2f}%")
    print(f"  Sharpe   : {m['sharpe']:7.2f}")
    print(f"  MDD      : {m['mdd']*100:7.2f}%")
    return m


### 기본 파라미터로 전체 기간 백테스트 (원본 재현)

In [ ]:
eq_base = run_backtest(df, ma_window=20, min_window=5, holding_days=3, slippage=0.004)

# 벤치마크: Buy & Hold
bh = df['close'] / df['close'].iloc[0] * 1_000_000

summary('Strategy (전체기간)', eq_base)
print('-' * 40)
summary('Buy & Hold', bh)

plt.plot(eq_base.index, eq_base.values, 'k', label='Strategy')
plt.plot(bh.index, bh.values, 'r', alpha=0.7, label='Buy & Hold')
plt.title('Strategy vs Buy & Hold (Full Period)')
plt.ylabel('Portfolio Value'); plt.legend(); plt.show()


---
## 3. 응용 실습과제 (기본기 → 응용)

이제 함수가 생겼으니 **파라미터/로직을 바꿔가며 실험**할 수 있습니다.
아래 과제들을 직접 코드로 채워 보세요. (각 셀의 `# TODO` 부분)

> 💡 정답을 하나로 못박기보다, **"결과가 어떻게/왜 달라지는지 해석"** 하는 것이 핵심입니다.


### 실습 3-1. 파라미터 민감도 분석

`holding_days`(보유일수)를 1, 3, 5, 10, 20으로 바꾸며 성과가 어떻게 변하는지 표로 정리하세요.
**"하나의 최고값"이 아니라 "안정적인 구간"** 이 있는지 관찰하는 것이 목적입니다.


In [ ]:
# 실습 3-1 예시 코드
rows = []
for hd in [1, 3, 5, 10, 20]:
    eq = run_backtest(df, holding_days=hd)
    m = performance(eq)
    rows.append({'holding_days': hd, **m})

result_31 = pd.DataFrame(rows).set_index('holding_days')
print(result_31.round(3))

# TODO: holding_days 대신 ma_window(5,10,20,60,120)로도 같은 분석을 해보세요.


### 실습 3-2. 손절(Stop-Loss) 추가

원본 전략에는 **손절 로직이 없습니다.** 평단가 대비 일정 % 이상 하락하면 즉시 매도하도록
`run_backtest` 를 변형한 `run_backtest_sl()` 을 만들고, 손절 유무의 MDD 차이를 비교하세요.


In [ ]:
def run_backtest_sl(df, ma_window=20, min_window=5, holding_days=3,
                    stop_loss=0.05, slippage=0.004, init_cash=1_000_000,
                    start=None, end=None):
    data = df.copy()
    data['ma'] = data['close'].rolling(ma_window).mean()
    data['roll_min'] = data['close'].rolling(min_window).min()
    sim = data.loc[start:end]

    cash, position, holding, avg_price = init_cash, 0, 0, 0
    equity = []
    for _, row in sim.iterrows():
        price = row['close']
        if not (np.isnan(row['ma']) or np.isnan(row['roll_min'])):
            if (price < row['ma']) and (price == row['roll_min']) and cash > price:
                position += 1; cash -= price; avg_price = price; holding = 0
            # TODO: 손절 조건을 완성하세요.
            #   힌트) position > 0 이고 price <= avg_price * (1 - stop_loss) 이면 전량 매도
            elif position > 0 and holding == holding_days:
                cash += position * price * (1 - slippage); position = 0
        if position > 0:
            holding += 1
        equity.append(cash + position * price)
    return pd.Series(equity, index=sim.index)

# TODO: stop_loss=0.05 로 실행 후 원본과 MDD/CAGR 비교


### 실습 3-3. 거래비용 민감도

`slippage` 를 0%, 0.2%, 0.4%, 1.0% 로 바꾸며 **CAGR이 얼마나 깎이는지** 확인하세요.
> 실전에서는 슬리피지 + 수수료 + 세금(매도 시 거래세)까지 반영해야 합니다.
> **거래가 잦은 전략일수록 비용에 취약**하다는 점을 수치로 체감해 보세요.


In [ ]:
# 실습 3-3 예시
for s in [0.0, 0.002, 0.004, 0.01]:
    m = performance(run_backtest(df, slippage=s))
    print(f"slippage {s*100:.1f}% -> CAGR {m['cagr']*100:6.2f}% | Sharpe {m['sharpe']:.2f}")


### 실습 3-4. (도전) 멀티 종목으로 확장

`fdr.DataReader` 로 종목 여러 개(예: 000660 SK하이닉스, 005380 현대차)를 받아
각각 백테스트한 뒤 **동일가중 합산 포트폴리오**의 성과를 구해 보세요.
단일 종목 대비 MDD가 줄어드는지(분산 효과) 확인하는 것이 포인트입니다.


In [ ]:
# 실습 3-4 스켈레톤
tickers = ['005930', '000660', '005380']
equities = {}
for t in tickers:
    d = fdr.DataReader(t, '2015').rename(columns=str.lower)
    equities[t] = run_backtest(d[['close']], init_cash=1_000_000)

# TODO: 세 equity curve를 날짜 기준 정렬 후 동일가중 합산하여 포트폴리오 성과 계산
# port = pd.concat(equities, axis=1).ffill().sum(axis=1)
# summary('3-Stock Portfolio', port)


---
## 4. 핵심 개념: In-Sample / Out-of-Sample 과 과최적화

### 4-1. 왜 "전체 기간 최적화"는 위험한가?

전략을 만들 때 흔히 이렇게 합니다.

> "여러 파라미터 조합을 **전체 기간**에 돌려보고, 가장 수익률 좋은 걸 고른다."

이건 **시험 문제를 미리 보고 답을 외우는 것(curve-fitting)** 과 같습니다.
과거에 가장 잘 맞은 파라미터가 미래에도 최적이라는 보장이 전혀 없습니다.
오히려 **과거의 노이즈(우연)에 맞춰졌을** 가능성이 큽니다. 이것이 **과최적화(Overfitting)** 입니다.

### 4-2. 해결의 출발점: 데이터를 나눈다

| 구분 | 이름 | 역할 |
|------|------|------|
| **IS** | In-Sample (학습/최적화 구간) | 파라미터를 **찾는** 데 사용 |
| **OOS** | Out-of-Sample (검증 구간) | 찾은 파라미터를 **처음 보는 데이터로 검증** |

**IS에서 고른 파라미터를, IS 성과가 아니라 OOS 성과로 평가**하는 것이 핵심입니다.
OOS에서 성과가 무너지면 → 그 전략은 **과최적화된 것**입니다.


### 4-3. 시연: IS에서 '최적' 파라미터를 찾고 OOS에서 검증

먼저 그리드 서치 함수를 만들고, 데이터를 앞(IS)/뒤(OOS)로 나눠 봅니다.


In [ ]:
def grid_search(df, ma_list, min_list, hold_list, start, end, metric='sharpe'):
    """[start, end] 구간에서 파라미터 조합을 전수 탐색, metric 기준 내림차순 정렬."""
    rows = []
    for ma, mn, hd in itertools.product(ma_list, min_list, hold_list):
        eq = run_backtest(df, ma_window=ma, min_window=mn, holding_days=hd,
                          start=start, end=end)
        rows.append({'ma': ma, 'min': mn, 'hold': hd, **performance(eq)})
    return pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)


# 탐색할 파라미터 그리드
MA_LIST   = [5, 10, 20, 60]
MIN_LIST  = [3, 5, 10]
HOLD_LIST = [1, 3, 5, 10]

# IS / OOS 분리 (앞 70% 학습, 뒤 30% 검증)
split = df.index[int(len(df) * 0.7)]
print(f"IS : {df.index[0].date()} ~ {split.date()}")
print(f"OOS: {split.date()} ~ {df.index[-1].date()}")


In [ ]:
# 1) IS 구간에서 '최적' 파라미터 탐색
is_result = grid_search(df, MA_LIST, MIN_LIST, HOLD_LIST,
                        start=None, end=split, metric='sharpe')
print('=== IS 상위 5개 조합 ===')
print(is_result.head().round(3))

best = is_result.iloc[0]
print(f"\n선택된 최적 파라미터: ma={int(best.ma)}, min={int(best['min'])}, hold={int(best.hold)}")


In [ ]:
# 2) 같은 파라미터를 IS와 OOS에 각각 적용해 성과 비교
p = dict(ma_window=int(best.ma), min_window=int(best['min']), holding_days=int(best.hold))

eq_is  = run_backtest(df, **p, start=None,  end=split)
eq_oos = run_backtest(df, **p, start=split, end=None)

print('>>> IS(학습 구간) 성과 — 당연히 좋아 보임')
summary('IS', eq_is)
print('-' * 40)
print('>>> OOS(검증 구간) 성과 — 진짜 실력')
summary('OOS', eq_oos)


> **관찰 포인트**
> IS에서 Sharpe가 가장 높았던 파라미터를 OOS에 적용하면 성과(Sharpe/CAGR)가 **눈에 띄게 하락**하는 경우가 많습니다.
> IS 성과와 OOS 성과의 **격차가 클수록 과최적화가 심한 것**입니다.
> "IS에서만 좋은 전략"은 실전에 배포하면 안 됩니다.


---
## 5. 워크포워드 분석 (Walk-Forward Analysis)

### 5-1. 한 번의 IS/OOS 분리로는 부족하다

4장의 방법은 좋지만, **딱 한 번** 나눈 것이라 운(luck)의 영향이 큽니다.
하필 OOS 구간이 대세 상승장이면 아무 전략이나 잘 나오고, 하락장이면 다 나쁘게 나옵니다.

**워크포워드**는 이 IS→OOS 검증을 **시간 축을 따라 여러 번 반복**합니다.

```
[  IS #1  ][OOS#1]
        [  IS #2  ][OOS#2]
                [  IS #3  ][OOS#3]
                        [  IS #4  ][OOS#4] ...
```

각 단계에서:
1. **IS 구간**에서 최적 파라미터를 찾고
2. 그 파라미터를 **바로 뒤 OOS 구간**에 적용해 성과를 기록한 뒤
3. 창(window)을 OOS 길이만큼 앞으로 밀어 반복

그리고 **모든 OOS 구간을 이어붙인 곡선** — 이것이 실전에 가장 가까운 성과 추정치입니다.
(주기적으로 파라미터를 재조정하며 굴리는 실제 운용을 그대로 흉내 낸 것)


In [ ]:
def walk_forward(df, ma_list, min_list, hold_list,
                 is_len=500, oos_len=125, metric='sharpe', init_cash=1_000_000):
    """롤링 워크포워드 분석.

    is_len  : In-Sample 길이(거래일). 500 ≈ 2년
    oos_len : Out-of-Sample 길이(거래일). 125 ≈ 6개월
    Returns
    -------
    wf_equity : pd.Series          # 이어붙인 OOS equity curve
    log       : pd.DataFrame       # 각 fold에서 선택된 파라미터/성과
    """
    idx = df.index
    n = len(df)
    segments, log = [], []
    i = 0
    cash = init_cash
    while i + is_len + oos_len <= n:
        is_start,  is_end  = idx[i],          idx[i + is_len - 1]
        oos_start, oos_end = idx[i + is_len],  idx[i + is_len + oos_len - 1]

        # 1) IS에서 최적 파라미터
        res = grid_search(df, ma_list, min_list, hold_list, is_start, is_end, metric)
        b = res.iloc[0]

        # 2) OOS에 적용 (자본은 이전 fold에서 이어받음)
        eq = run_backtest(df, ma_window=int(b.ma), min_window=int(b['min']),
                          holding_days=int(b.hold), init_cash=cash,
                          start=oos_start, end=oos_end)
        cash = eq.iloc[-1]
        segments.append(eq)

        m = performance(eq)
        log.append({'oos_start': oos_start.date(), 'oos_end': oos_end.date(),
                    'ma': int(b.ma), 'min': int(b['min']), 'hold': int(b.hold),
                    'oos_cagr': m['cagr'], 'oos_sharpe': m['sharpe'], 'oos_mdd': m['mdd']})
        i += oos_len

    wf_equity = pd.concat(segments)
    return wf_equity, pd.DataFrame(log)


In [ ]:
wf_equity, wf_log = walk_forward(df, MA_LIST, MIN_LIST, HOLD_LIST,
                                 is_len=500, oos_len=125, metric='sharpe')

print('=== 각 OOS 구간에서 선택된 파라미터와 성과 ===')
print(wf_log.round(3).to_string(index=False))
print()
summary('Walk-Forward (이어붙인 OOS)', wf_equity)


### 5-2. 결정적 비교: 전체최적화(과최적화) vs 워크포워드(정직한 추정)

같은 파라미터 그리드로,
- **(A) 전체 기간 최적화**: 전체 데이터에서 가장 좋은 파라미터 하나를 골라 전체에 적용 → *실전에서는 불가능한 "미래를 아는" 성과*
- **(B) 워크포워드 OOS**: 매 시점 과거만 보고 파라미터를 정한 성과 → *실전에서 실제로 얻었을 성과*

두 곡선의 격차가 바로 **"백테스트 낙관 편향(optimism bias)"** 의 크기입니다.


In [ ]:
# (A) 전체 기간 최적화 — 미래를 훔쳐본 결과
full_best = grid_search(df, MA_LIST, MIN_LIST, HOLD_LIST,
                        start=None, end=None, metric='sharpe').iloc[0]
eq_full = run_backtest(df, ma_window=int(full_best.ma), min_window=int(full_best['min']),
                       holding_days=int(full_best.hold))

# 비교 구간을 워크포워드 OOS 시작점에 맞춤
eq_full_aligned = eq_full.loc[wf_equity.index[0]:]
eq_full_aligned = eq_full_aligned / eq_full_aligned.iloc[0] * 1_000_000
wf_norm = wf_equity / wf_equity.iloc[0] * 1_000_000

print('>>> (A) 전체최적화 = 과최적화된 "이상적" 성과')
summary('Full-Sample Optimized', eq_full_aligned)
print('-' * 40)
print('>>> (B) 워크포워드 = 정직한 성과 추정')
summary('Walk-Forward OOS', wf_norm)

plt.plot(eq_full_aligned.index, eq_full_aligned.values, 'r', label='(A) Full-Sample Optimized (overfit)')
plt.plot(wf_norm.index, wf_norm.values, 'k', label='(B) Walk-Forward OOS (honest)')
plt.title('Overfitted Backtest vs Walk-Forward'); plt.ylabel('Portfolio Value')
plt.legend(); plt.show()


> **해석**
> (A) 빨간 곡선(전체최적화)이 (B) 검은 곡선(워크포워드)보다 훨씬 좋아 보인다면,
> 그 차이는 **실력이 아니라 미래를 훔쳐본 대가**입니다.
> 실전에 배포하면 우리가 실제로 얻는 것은 (B)에 가깝습니다.
>
> **워크포워드 성과가 벤치마크(Buy&Hold) 대비, 그리고 거래비용 반영 후에도 살아남아야**
> 비로소 "쓸 만한 전략"이라고 말할 수 있습니다.


### 5-3. 파라미터 안정성(Parameter Stability) 관찰

`wf_log` 를 보면 fold마다 선택된 파라미터가 **얼마나 흔들리는지** 알 수 있습니다.
- fold마다 파라미터가 **크게 요동친다** → 전략이 데이터에 과민(불안정), 신뢰도 낮음
- fold가 달라도 **비슷한 파라미터가 반복 선택** → 견고한(robust) 신호일 가능성

좋은 전략은 "특정 마법의 숫자"가 아니라 **넓은 파라미터 구간에서 고르게 잘 되는** 전략입니다.


In [ ]:
# fold별 선택 파라미터 변동 시각화
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, col in zip(axes, ['ma', 'min', 'hold']):
    ax.plot(range(len(wf_log)), wf_log[col], 'o-')
    ax.set_ylabel(col)
axes[0].set_title('Selected Parameters per Walk-Forward Fold')
axes[-1].set_xlabel('Fold #')
plt.show()


---
## 6. 심화 실습과제

1. **Anchored(고정 시작점) 워크포워드 구현**
   - 위 `walk_forward` 는 IS 창이 앞으로 밀리는 **Rolling** 방식입니다.
   - IS 시작점을 고정하고 창을 계속 **늘려가는** Anchored 방식으로 바꿔, 두 방식의 OOS 성과를 비교하세요.

2. **`is_len` / `oos_len` 민감도**
   - `is_len` 을 250/500/750, `oos_len` 을 63/125/250으로 바꿔가며 워크포워드를 돌리고,
     OOS 성과가 이 설정에 얼마나 민감한지 표로 정리하세요.
   - *한 설정에서만 잘 나온다면 그 자체가 위험 신호입니다.*

3. **파라미터 안정성 히트맵**
   - 특정 IS 구간에서 (ma × hold) 조합별 Sharpe를 2D 히트맵으로 그려,
     "최적점"이 **뾰족한 봉우리인지 넓은 고원인지** 확인하세요. (고원이 더 신뢰할 만함)

4. **거래비용을 반영한 워크포워드**
   - `run_backtest` 의 `slippage` 를 현실적으로 높인 뒤(예: 0.5%),
     워크포워드 OOS 성과가 벤치마크를 여전히 이기는지 검증하세요.

5. **다른 전략으로 교체**
   - 매수 조건을 평균회귀 대신 **추세추종(20일 신고가 돌파 매수)** 으로 바꾸고,
     동일한 IS/OOS·워크포워드 파이프라인으로 검증하세요.
   - *파이프라인은 그대로 두고 전략만 갈아끼우는* 모듈화의 이점을 체감할 수 있습니다.


---
## 7. 핵심 요약

- **백테스팅 코드는 함수화**해야 파라미터 실험·검증이 가능하다.
- 예쁜 전체기간 수익곡선은 **과최적화의 산물**일 수 있다 — 반드시 의심하라.
- **In-Sample / Out-of-Sample 분리**: 파라미터는 IS에서 찾고, 평가는 OOS로 한다.
- **워크포워드 분석**: IS→OOS 검증을 시간축으로 반복해 이어붙인 OOS 곡선이 **실전에 가장 가까운 성과**다.
- 좋은 전략의 조건: **① OOS에서 살아남고 ② 거래비용 후에도 벤치마크를 이기며 ③ 파라미터가 안정적**이다.

> "In-Sample에서 잘 되는 전략은 널렸다. **Out-of-Sample에서 살아남는 전략만이 돈을 번다.**"
